# Wakil AI — Phase 2: A Real Project + Overfitting Lab

In Phase 1, everything lived in one notebook. Now the same neural network lives in a **clean Python project** (`wakil_ai/` package + command-line scripts + automated tests), and this notebook just *uses* it.

**What you will do here**
1. Download the project from GitHub and run its **automated tests**.
2. Train, evaluate and predict with **command-line scripts** (the way real ML projects work).
3. Run the **overfitting lab**: watch a model memorize instead of learn, then fight it with **L2 regularization**, **early stopping** and **more data**.

Run the cells in order (**Runtime → Run all** works too). Total time: about 2–3 minutes on free Colab CPU.

## Cell 1 — Get the project

Downloads (clones) the repository from GitHub into Colab and moves into its folder. Nothing is saved to your Google Drive.

**Expected output:** a path ending in `/WAKIL-AI`.

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/yassine1158/WAKIL-AI.git"

if not os.path.exists("wakil_ai/model.py"):   # not inside the project folder yet
    if not os.path.exists("WAKIL-AI"):
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir("WAKIL-AI")
sys.path.insert(0, os.getcwd())              # so that "import wakil_ai" finds the project
print("Working directory:", os.getcwd())

## Cell 2 — Install requirements and run the automated tests

**Why tests?** In Phase 1 you checked things by looking at outputs. A real project checks itself automatically: every time the code changes, `pytest` re-runs checks such as:
- the **gradient check** (backpropagation is mathematically correct, with 1 or 2 hidden layers, with and without L2),
- softmax rows sum to 1, the untrained loss is ≈ ln(10),
- training actually learns a simple problem,
- early stopping stops and restores the best epoch,
- save → load gives identical predictions (also for your Phase 1 `.npz` file).

On GitHub, the same tests run automatically on every push (see the **Actions** tab of the repository).

**Expected output:** `25 passed` (15 tests for the digit classifier, 10 for the language model added in Phase 3).

In [ ]:
!pip install -q -r requirements.txt
!python -m pytest -q

## Cell 3 — Train from the command line

`train.py` does exactly what Phase 1's training cells did, but as a reusable program with options:

```
python train.py --hidden 128 --lr 0.1 --batch-size 64 --epochs 20 --seed 42
```

New in Phase 2: the model keeps the **best checkpoint** — the epoch with the lowest validation loss — instead of simply the last epoch.

Notice that `train.py` **never touches the test set**. Only `evaluate.py` does. Keeping them in separate programs makes it impossible to "accidentally" tune on the test set.

**Expected output** (same numbers as Phase 1, because the seed is the same — that is **reproducibility**):
```
Epoch  19/20 | train loss 0.0254 acc 99.50% | val loss 0.0796 acc 97.74% | ...  *best
Epoch  20/20 | train loss 0.0260 acc 99.46% | val loss 0.0813 acc 97.54% | ...
Restored best model from epoch 19 (val loss 0.0796).
```

In [ ]:
!python train.py --hidden 128 --lr 0.1 --batch-size 64 --epochs 20 --seed 42

from IPython.display import Image, display
display(Image("outputs/training_curves.png"))

## Cell 4 — Evaluate on the unseen test set

**Expected output:** `TEST accuracy: 97.78%  (9778 / 10000 correct)`, a per-digit report, the confusion matrix and two grids of test images (random ones, and mistakes).

In [ ]:
!python evaluate.py --model models/wakil_ai_mnist.npz

for figure in ["confusion_matrix.png", "test_predictions.png", "test_mistakes.png"]:
    display(Image(f"outputs/{figure}"))

## Cell 5 — Predict

`predict.py` is the "product": give it images, get digits. It works on MNIST test images (`--test-index`) or on your own photos (`--image my_digit.jpg` — upload the file with the 📁 panel on the left first).

**Expected output:**
```
test image #0: predicted 7 (100.0%) | true: 7 OK | top 3 -> 7: 100.0%, 3: 0.0%, 9: 0.0%
...
test image #8: predicted 5 (67.6%) | true: 5 OK | top 3 -> 5: 67.6%, 6: 27.4%, 8: 2.6%
```
Look at image #8: the model is right but **unsure** (it hesitates with 6). The probabilities tell you how much to trust each answer.

In [ ]:
!python predict.py --test-index 0 1 2 8

---
# 🧪 The Overfitting Lab

## Overfitting vs. generalization, in plain words

Imagine a student preparing for an exam with **10 past exam papers**.
- **Generalization:** the student understands the method, so they can solve *new* questions.
- **Overfitting:** the student memorized the 10 answer sheets by heart. Perfect score on those papers… and fails the real exam.

A neural network with 101,770 parameters can easily *memorize* a small training set. The symptoms:
- **training** accuracy → 100% and training loss → 0,
- **validation** loss stops improving and then **gets worse**,
- a big **gap** between train and validation accuracy.

This is exactly why we judge a model **only on data it never saw** (validation during development, test once at the end). Training accuracy measures memory; test accuracy measures intelligence.

## Experiment 1 — Provoke overfitting: train on only 1,000 images

Same network, but only 1,000 training images instead of 50,000, for 150 epochs.

**Expected output:** train accuracy reaches **100%** while validation accuracy stays around **87–88%**. Validation loss reaches its minimum around epoch 33, then climbs back up: from that point, the model is memorizing.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from wakil_ai import TrainConfig, WakilNet, evaluate, load_prepared_mnist, train

small = load_prepared_mnist(train_limit=1000)
print("Training images:", len(small["X_train"]), "| validation images:", len(small["X_val"]))


def run(config, data=small):
    model = WakilNet(seed=42)
    history = train(model, data["X_train"], data["y_train"], data["X_val"], data["y_val"],
                    config, verbose=False)
    return model, history


model_plain, hist_plain = run(TrainConfig(epochs=150))
best = hist_plain["best_epoch"]
print(f"Final train accuracy:      {hist_plain['train_acc'][-1]:.2%}")
print(f"Final validation accuracy: {hist_plain['val_acc'][-1]:.2%}")
print(f"Validation loss: best {min(hist_plain['val_loss']):.3f} at epoch {best}, "
      f"final {hist_plain['val_loss'][-1]:.3f}  <- it went back up: overfitting")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
epochs = range(1, 151)
axes[0].plot(epochs, hist_plain["train_loss"], label="train")
axes[0].plot(epochs, hist_plain["val_loss"], label="validation")
axes[0].axvline(best, color="gray", linestyle="--", label=f"best epoch ({best})")
axes[0].set(xlabel="epoch", ylabel="loss", title="Loss: validation goes back up", yscale="log")
axes[1].plot(epochs, hist_plain["train_acc"], label="train")
axes[1].plot(epochs, hist_plain["val_acc"], label="validation")
axes[1].set(xlabel="epoch", ylabel="accuracy", title="Accuracy: 100% train vs ~87% validation")
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()

## Experiment 2 — Three cures

### Cure A: L2 regularization (weight decay)
We add a penalty for large weights to the loss:
$$L_{\text{total}} = L_{\text{cross-entropy}} + \frac{\lambda}{2}\sum W^2$$
Its gradient is simply $\lambda W$, so every update also shrinks each weight a little (`dW += weight_decay * W` in `model.py`). Big weights let a network build very "spiky" functions that fit individual training images; small weights force **smoother** functions that generalize better. $\lambda$ (`weight_decay`) controls the strength: too small = no effect, too large = the model can't learn (underfitting).

### Cure B: Early stopping
Watch the validation loss, and **stop when it hasn't improved for `patience` epochs**, keeping the best checkpoint. Cheap, simple, and used almost everywhere.

### Cure C: More data
The most powerful cure of all. With more examples, memorizing becomes harder than learning the real pattern.

**Expected output:** a table similar to this (our reference run):

| Setup | Epochs run | Val loss (best model) | Val accuracy (best model) |
|---|---|---|---|
| 1,000 images, no cure | 150 | 0.415 | 87.5% |
| 1,000 images + L2 (λ=0.01) | 150 | 0.400 | 88.1% |
| 1,000 images + early stopping | 43 | 0.415 | 87.5% |
| **50,000 images** (Phase 2 model) | 20 | 0.080 | **97.7%** |

**The honest lesson:** L2 keeps the validation loss from getting worse, and early stopping gets the same result as the full run in less than a third of the time. Both help a little. But **more data** wins by far, which is why big AI labs spend so much on data.

In [ ]:
model_l2, hist_l2 = run(TrainConfig(epochs=150, weight_decay=0.01))
model_es, hist_es = run(TrainConfig(epochs=150, patience=10))
big_model, big_meta = WakilNet.load("models/wakil_ai_mnist.npz")

rows = [("1,000 images, no cure", hist_plain, model_plain),
        ("1,000 images + L2 (λ=0.01)", hist_l2, model_l2),
        ("1,000 images + early stopping", hist_es, model_es)]
print(f"{'Setup':<34}{'Epochs':>7}{'Val loss':>10}{'Val acc':>9}")
for name, hist, model in rows:
    val_loss, val_acc = evaluate(model, small["X_val"], small["y_val"])
    print(f"{name:<34}{len(hist['val_loss']):>7}{val_loss:>10.3f}{val_acc:>9.2%}")
val_loss, val_acc = evaluate(big_model, small["X_val"], small["y_val"])
print(f"{'50,000 images (Phase 2 model)':<34}{20:>7}{val_loss:>10.3f}{val_acc:>9.2%}")

plt.figure(figsize=(8, 4))
plt.plot(hist_plain["val_loss"], label="no cure")
plt.plot(hist_l2["val_loss"], label="L2 regularization (λ=0.01)")
plt.plot(hist_es["val_loss"], label="early stopping (patience 10)", linewidth=3)
plt.xlabel("epoch")
plt.ylabel("validation loss")
plt.title("Validation loss with 1,000 training images")
plt.legend()
plt.show()

## Your turn (pick at least one)
1. Try `weight_decay=0.1` in Experiment 2. What happens? (Hint: **underfitting** — too much regularization.)
2. Run `!python train.py --train-limit 5000 --epochs 60` and compare its best validation accuracy to 1,000 and 50,000 images. Plot "accuracy vs. number of training images".
3. Try a deeper network: `!python train.py --hidden 256 128 --epochs 20`, then `!python evaluate.py`. Is it better? Is it worth the extra parameters?

## Phase 2 success checklist
- [ ] Cell 2 shows `25 passed`
- [ ] Cell 3 reproduces the reference numbers (best val accuracy about 97.7% at epoch 19; the last decimals can vary slightly between computers)
- [ ] Cell 4 shows a test accuracy ≥ 97%
- [ ] Cell 5 predicts test images correctly
- [ ] In Experiment 1 you can point to the epoch where overfitting starts, and explain why
- [ ] You can explain in your own words: overfitting, generalization, L2, early stopping, and why we only trust test accuracy
- [ ] The GitHub **Actions** tab shows a green ✓ for the latest commit

**Next: Phase 3** — a tiny character-level language model (tokenizer, embeddings, next-character prediction), then a small Transformer with attention.